In [ ]:
# accuracy in Pytorch
correct = 0
total = 0

correct += (
    predictions == y
).sum().item()

total += y.numel()

accuracy = correct / total

In [ ]:
# evaluation tools
from sklearn.metrics import(
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    roc_curve,
    roc_auc_score
)

import torch

# model predictions on entire test set
def get_predictions(
    model,
    data_loader,
    device
):

    model.eval()

    probabilities = []
    predictions = []
    targets = []

    with torch.no_grad():

        for X, y in data_loader:

            X = X.to(device)

            logits = model(X)

            probs = torch.sigmoid(
                logits
            )

            preds = (
                probs >= 0.5
            ).float()


            probabilities.extend(
                probs.cpu().numpy().flatten()
            )

            predictions.extend(
                preds.cpu().numpy().flatten()
            )

            targets.extend(
                y.numpy().flatten()
            )

    return (
        targets,
        predictions,
        probabilities
    )

In [ ]:
y_true, y_pred, y_prob = get_predictions(
    model,
    test_loader,
    device

)

# confusion matrix
cm = confusion_matrix(
    y_true,
    y_pred
)

print(cm)

In [ ]:
# visuaizing the confusion matrix
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred
)

plt.title("Confusion Matrix")
plt.show()

In [ ]:
# accuracy score
accuracy = accuracy_score(
    y_true,
    y_pred
)

print(
    f"Accuracy: {accuracy:.4f}"
)

In [ ]:
# precision score
precision = precision_score(
    y_true,
    y_pred
)

print(
    f"Precision: {precision:.4f}"
)

In [ ]:
# recall score
recall = recall_score(
    y_true,
    y_pred
)

print(
    f"Recall: {recall:.4f}"
)

In [ ]:
# F1 score
f1 = f1_score(
    y_true,
    y_pred
)

print(
    f"F1 Score: {f1:.4f}"
)

In [ ]:
# specifity
tn, fp, fn, tp = confusion_matrix(
    y_true,
    y_pred
).ravel()

specificity = tn / (
    tn + fp
)

print(
    f"Specificity: {specificity:.4f}"
)

In [ ]:
# classification report
print(
    classification_report(
        y_true,
        y_pred
    )
)

In [ ]:
# ROC Curve in Python
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(
    y_true,
    y_prob
)

# then:
plt.figure(figsize=(8, 5))

plt.plot(
    fpr,
    tpr,
    label="ROC Curve"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")

plt.legend()

plt.show()

In [ ]:
# roc-auc curve - Receiver Operating Characteristic
# auc - Area Under the ROC Curve
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(
    y_true,
    y_prob
)

print(
    f"ROC-AUC: {auc:.4f}"
)

In [ ]:
# threshold tuning
predictions = (
    probabilities >0.5
).float()

# then
threshold = 0.30

predictions = (
    y_prob >= threshold
).astype(int)

In [ ]:
# recalculate the threshold
precision_score(
    y_true,
    predictions
)

recall_score(
    y_true,
    predictions
)

In [ ]:
# comparing the threshold
thresholds = [
    0.2,
    0.3,
    0.4,
    0.5,
    0.6,
    0.7,
    0.8
]

for threshold in thresholds:

    predictions = (
        y_prob >= threshold
    ).astype(int)

    precision = precision_score(
        y_true,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        predictions,
        zero_division=0
    )

    print(
        f"Threshold={threshold:.1f} | "
        f"Precision={precision:.3f} | "
        f"Recall={recall:.3f} | "
        f"F1={f1:.3f}"
    )

# NB - Very useful in real-world projects

In [17]:
# reusable evaluation function
# evaluation logic

import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)


def evaluate_model(
    model,
    dataloader,
    device
):

    model.eval()

    y_true = []
    y_prob = []

    with torch.no_grad():

        for X, y in dataloader:

            X = X.to(device)

            logits = model(X)

            probabilities = torch.sigmoid(
                logits
            )

            y_true.extend(
                y.numpy().flatten()
            )

            y_prob.extend(
                probabilities
                .cpu()
                .numpy()
                .flatten()
            )

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)

    y_pred = (
        y_prob >= 0.5
    ).astype(int)

    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred
    ).ravel()

    # Metrics
    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    specificity = tn / (
        tn + fp
    )

    auc = roc_auc_score(
        y_true,
        y_prob
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "auc": auc,
        "confusion_matrix": np.array([
            [tn, fp],
            [fn, tp]
        ]),
        "y_true": y_true,
        "y_prob": y_prob,
        "y_pred": y_pred
    }

In [ ]:
# Using the Evaluation Function

# After loading the best model
metrics = evaluate_model(
    model,
    test_loader,
    device
)

# then
print(
    f"Accuracy: "
    f"{metrics['accuracy']:.4f}"
)

print(
    f"Precision: "
    f"{metrics['precision']:.4f}"
)

print(
    f"Recall: "
    f"{metrics['recall']:.4f}"
)

print(
    f"Specificity: "
    f"{metrics['specificity']:.4f}"
)

print(
    f"F1: "
    f"{metrics['f1']:.4f}"
)

print(
    f"ROC-AUC: "
    f"{metrics['auc']:.4f}"
)

# confusion matrix
print(
    metrics["confusion_matrix"]
)

In [ ]:
# generate the classification report
print(
    classification_report(
        metrics["y_true"],
        metrics["y_pred"],
        target_names=[
            "Class 0",
            "Class 1"
        ],
        zero_division=0
    )
)

In [ ]:
# ROC Curve from Our Evaluation Results
fpr, tpr, _ = roc_curve(
    metrics["y_true"],
    metrics["y_prob"]
)

plt.figure(figsize=(8, 5))

plt.plot(
    fpr,
    tpr,
    label=f"AUC = {metrics['auc']:.3f}"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")

plt.legend()

plt.show()

##### Evaluation Pipeline
                  TEST DATA
                      │
                      ↓
                    Model
                      │
                      ↓
                    Logits
                      │
                 Sigmoid()
                      │
                      ↓
                Probabilities
                      │
             ┌────────┴────────┐
             ↓                 ↓
       Threshold 0.5       ROC Analysis
             ↓                 ↓
       Predictions           AUC
             │
             ↓
      Confusion Matrix
             │
    ┌────────┼─────────┐
    ↓        ↓         ↓
Precision  Recall     F1
    │        │         │
    └────────┼─────────┘
             ↓
        Model Report

🧪 PRACTICAL PROJECT

Now let's apply Lesson 6 to the model from Lesson 5.

After training:

model.load_state_dict(
    torch.load(
        "best_model.pth",
        map_location=device,
        weights_only=True
    )
)

Then:

metrics = evaluate_model(
    model,
    test_loader,
    device
)

Print:

Accuracy
Precision
Recall
Specificity
F1
ROC-AUC

Then display:

Confusion Matrix
Classification Report
ROC Curve

🧪 LESSON 6 EXERCISES
Exercise 1 — Confusion Matrix

Given:

TP = 80
TN = 850
FP = 30
FN = 40

Calculate:

Accuracy
Precision
Recall
Specificity
F1 Score
Exercise 2 — Interpret the Model

Suppose:

Accuracy    = 95%
Precision   = 82%
Recall      = 60%
Specificity = 98%
F1          = 69%
AUC         = 91%

Would you consider this model good?

More importantly:

What kind of application might make this model problematic despite the 95% accuracy?

Think about the relatively low recall.

🧪 Exercise 3 — Threshold Experiment

Using your trained model, calculate:

Threshold = 0.2
Threshold = 0.3
Threshold = 0.4
Threshold = 0.5
Threshold = 0.6
Threshold = 0.7
Threshold = 0.8

For each threshold calculate:

Precision
Recall
F1

Create a table.

Determine which threshold gives you the highest F1 score.

🧪 Exercise 4 — Imbalanced Dataset

Create:

X, y = make_classification(
    n_samples=5000,
    n_features=10,
    weights=[0.95, 0.05],
    random_state=42
)

Train your model.

Then compare:

Accuracy
Precision
Recall
F1
ROC-AUC

Ask yourself:

Is accuracy still enough?

🏆 LESSON 6 FINAL CHALLENGE

Build a reusable evaluation module:

project/
│
├── data/
│
├── models/
│   └── model.py
│
├── engine/
│   └── engine.py
│
├── evaluation/
│   └── metrics.py
│
├── plots/
│
├── train.py
│
└── evaluate.py


Your metrics.py should contain:

get_predictions()
evaluate_model()

and calculate:

Accuracy
Precision
Recall
Specificity
F1
ROC-AUC
Confusion Matrix

Your evaluate.py should:

1. Load best_model.pth
2. Load the test DataLoader
3. Generate predictions
4. Print all metrics
5. Display the confusion matrix
6. Display the ROC curve
7. Print the classification report
8. Save the evaluation results

For example:

evaluation_results.csv

containing:

metric,value
accuracy,0.94
precision,0.91
recall,0.89
specificity,0.96
f1,0.90
roc_auc,0.97

## Entire Deep Learning Workflow

                 DATA
                  ↓
              DataLoader
                  ↓
             Neural Network
                  ↓
               Logits
                  ↓
                 Loss
                  ↓
             Backpropagation
                  ↓
              Optimizer
                  ↓
           Updated Weights
                  ↓
              Validation
                  ↓
          Best Model Selection
                  ↓
               Test Set
                  ↓
             Probabilities
                  ↓
        ┌─────────┴─────────┐
        ↓                   ↓
  Classification        ROC Analysis
     Metrics                ↓
        ↓                  AUC
 ┌──────┼──────┐
 ↓      ↓      ↓
P      R      F1
        ↓
 Confusion Matrix
        ↓
 Final Evaluation